# Minimal YOLO — live twin JPEG → predictions + overlay

This walkthrough parallels **[`model_predict_on_twin.py`](model_predict_on_twin.py)**:

1. **Fetch JPEG** · `Twin.get_latest_frame()`
2. **Predict** · `Cyberwave.models.load(...).predict(..., confidence=CONF)`
3. **Tables** · `PredictionResult.describe_detections_text()`
4. **Overlay** · Ultralytics `Results.plot(pil=True, img=BGR)`, then **`matplotlib`** (same RGB↔BGR pattern as **`yolo.ipynb`**)

---

### Before you begin

| Check | Detail |
|:-----|:-------|
| **Auth** | Prefer `cyberwave login --token YOUR_TOKEN`; credentials land in `~/.cyberwave/credentials.json`. |
| **Kernel** | After login, restart the kernel **or** run the credential importer cell. |
| **Twin** | `TWIN_ID` may be UUID **or** `workspace-slug/twins/entity-slug`. |
| **Camera + driver** | Use a twin whose asset declares an **RGB camera** capability, keep **Cyberwave Edge Core** connected, and leave the **camera driver** publishing frames—otherwise **`Twin.get_latest_frame()`** fails or yields stale blanks. |

**Docs (official):** [Digital twins & sensors](https://docs.cyberwave.com/use-cyberwave/digital-twins) · [Camera get started / edge install](https://docs.cyberwave.com/hardware/camera/get-started) · [Edge overview](https://docs.cyberwave.com/edge/overview)


## Step 1 — Install packages

Includes **`opencv-python-headless`** so we can feed BGR tensors into Ultralytics overlays.

In [ ]:
%pip install -q "cyberwave[ml]" cyberwave-cli matplotlib pillow numpy opencv-python-headless requests

## Step 2 — Login (recommended: `--token`)

The docs flow expects **`cyberwave login --token`**, keeping secrets outside notebooks:

```bash
cyberwave login --token YOUR_API_TOKEN_HERE
```

Restart the Jupyter kernel afterward—or run Section 4—so **`os.environ`** receives **`CYBERWAVE_API_KEY`**.

> Email/password **`cyberwave login`** flows remain available on the CLI — use what your team mandates.

## Step 3 — Twin + model knobs

| Setting | Meaning |
|---------|---------|
| `TWIN_ID` | Twin UUID **or** full slug like `workspace/twins/my-cam`. |
|`MODEL_ID`| Default **`yolo26n.pt`** (matches `model_predict_on_twin.py`) |
|`CONF` | Passed through to **`LoadedModel.predict`** |

In [ ]:
TWIN_ID = "your-uuid-here"   # Paste UUID here

MODEL_ID = "yolo26n.pt"

CONF = 0.25

## Step 4 — Import CLI credentials into the kernel

Loads **`token`** → **`CYBERWAVE_API_KEY`** for `Cyberwave()` without committing secrets.

In [ ]:
import json
import os
from pathlib import Path

_cred_path = Path("~/.cyberwave/credentials.json").expanduser()
_token = json.loads(_cred_path.read_text())["token"]
os.environ["CYBERWAVE_API_KEY"] = _token


## Step 5 — Inference + textual summary + overlay figure

Plots appear inline via **`matplotlib`**. Prediction rows print first for quick CLI-style inspection.

In [ ]:
from __future__ import annotations

from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from cyberwave import Cyberwave
from cyberwave.models.types import PredictionResult

%matplotlib inline


def plot_ultralytics_result(
    pred: PredictionResult,
    title: str,
    *,
    orig_rgb: np.ndarray | None,
) -> None:
    """BGR base for Ultralytics draw, then PIL RGB for matplotlib."""
    if pred.raw is None or len(pred.raw) == 0:
        print("No Ultralytics raw Results — showing camera frame.")
        rgb = np.asarray(orig_rgb)
    else:
        import cv2

        plot_kw: dict[str, object] = {}
        if orig_rgb is not None:
            plot_kw["img"] = cv2.cvtColor(orig_rgb, cv2.COLOR_RGB2BGR)
        plotted = pred.raw[0].plot(pil=True, **plot_kw)
        if hasattr(plotted, "convert"):
            rgb = np.asarray(plotted.convert("RGB"))
        else:
            rgb = np.asarray(plotted)[..., ::-1]

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(rgb)
    ax.axis("off")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


_tid = str(TWIN_ID).strip()
if not _tid:
    raise ValueError("Set TWIN_ID in the configuration cell (UUID or unified slug like ws/twins/name).")

cw = Cyberwave()
twin = cw.twins.get(_tid)

jpeg_bytes = twin.get_latest_frame()
pil_image = Image.open(BytesIO(jpeg_bytes)).convert("RGB")
orig_rgb = np.asarray(pil_image, dtype=np.uint8)

model = cw.models.load(MODEL_ID)
pred = model.predict(pil_image, confidence=CONF)

print(pred.describe_detections_text())
plot_ultralytics_result(pred, f"{MODEL_ID} @ conf={CONF} · twin preview", orig_rgb=orig_rgb)
